## Importando bibliotecas e funções

In [4]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

# Diretório principal
ROOT_DIR = Path.cwd().parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Diretórios
DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PCA_DIR = PROCESSED_DIR / "pca"
RESULTS_DIR = ROOT_DIR / "results"

RESULTS_DIR.mkdir(exist_ok=True)

# Funções do projeto
from src.preprocessing import (
    load_clinical_data,
    map_patient_subtypes
)

from src.dimensionality_reduction import (
    run_pacmap,
    run_umap
)

from src.clustering import (
    run_hdbscan,
    run_leiden
)

from importlib import reload
import src.experiments as exp

reload(exp)

run_grid_search = exp.run_grid_search

In [5]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.ERROR)

## Carregando os dados

In [2]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

# Diretório principal
ROOT_DIR = Path.cwd().parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Diretórios
DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PCA_DIR = PROCESSED_DIR / "pca"
RESULTS_DIR = ROOT_DIR / "results"

RESULTS_DIR.mkdir(exist_ok=True)

# Funções do projeto
from src.preprocessing import (
    load_clinical_data,
    map_patient_subtypes
)

from src.dimensionality_reduction import (
    run_pacmap,
    run_umap
)

from src.clustering import (
    run_hdbscan,
    run_leiden
)

from src.experiments import (
    run_grid_search
)

from src.experiments import select_best_models

## PAM50

In [6]:
PATH_CLINICO = PROCESSED_DIR / "clinical_final.csv"

pacientes, dict_subtipos = load_clinical_data(
    PATH_CLINICO
)

print("Pacientes clínicos:", len(dict_subtipos))
print(pacientes["SUBTYPE"].value_counts())

Pacientes clínicos: 970
SUBTYPE
BRCA_LumA      494
BRCA_LumB      193
BRCA_Basal     171
BRCA_Her2       77
BRCA_Normal     35
Name: count, dtype: int64


## Grades de hiperparâmetros

In [7]:
# PaCMAP
param_grid_pacmap = {
    "n_neighbors": [10, 15, 30, 50],
    "MN_ratio": [0.5, 1.0],
    "FP_ratio": [2.0, 3.0]
}

# UMAP
param_grid_umap = {
    "n_neighbors": [15, 30, 50],
    "min_dist": [0.01, 0.1, 0.25]
}

# HDBSCAN
param_grid_hdbscan = {
    "min_cluster_size": [10, 15, 30, 40, 50, 75, 90],
    "min_samples": [1, 5, 10, 15, 20]
}

param_grid_leiden = {
    "n_neighbors": [10, 15, 30, 50],
    "resolution": [0.05, 0.10, 0.20, 0.3, 0.40, 0.60, 1.0]
}

# Organizar os espaços de busca
reduction_grids = {
    "PaCMAP": param_grid_pacmap,
    "UMAP": param_grid_umap
}

clustering_grids = {
    "HDBSCAN": param_grid_hdbscan,
    "Leiden": param_grid_leiden
}

## Importando as bases PCA 50 Componentes Principais

In [8]:
import pandas as pd

CENARIOS_PCA = {
    "MAD 3.000": "PCA_MAD 3.000.csv",
    "MAD 5.000": "PCA_MAD 5.000.csv",
    "MAD 10.000": "PCA_MAD 10.000.csv",
    "MAD 20.000": "PCA_MAD 20.000.csv"
}

datasets_pca = {}

for nome, arquivo in CENARIOS_PCA.items():

    caminho = PCA_DIR / arquivo

    if not caminho.is_file():
        print(f"Arquivo não encontrado: {caminho}")
        continue

    df = pd.read_csv(
        caminho,
        index_col=0
    )

    datasets_pca[nome] = df

    print(f"\n{nome}")
    print("Dimensões:", df.shape)

print("\nTotal de bases carregadas:", len(datasets_pca))


MAD 3.000
Dimensões: (970, 50)

MAD 5.000
Dimensões: (970, 50)

MAD 10.000
Dimensões: (970, 50)

MAD 20.000
Dimensões: (970, 50)

Total de bases carregadas: 4


In [9]:
labels_pam50 = {}

for nome, df in datasets_pca.items():

    y = map_patient_subtypes(
        df.index,
        dict_subtipos
    )

    labels_pam50[nome] = y

    print(f"\n{nome}")
    print(y.value_counts())


MAD 3.000
SUBTYPE
BRCA_LumA      494
BRCA_LumB      193
BRCA_Basal     171
BRCA_Her2       77
BRCA_Normal     35
Name: count, dtype: int64

MAD 5.000
SUBTYPE
BRCA_LumA      494
BRCA_LumB      193
BRCA_Basal     171
BRCA_Her2       77
BRCA_Normal     35
Name: count, dtype: int64

MAD 10.000
SUBTYPE
BRCA_LumA      494
BRCA_LumB      193
BRCA_Basal     171
BRCA_Her2       77
BRCA_Normal     35
Name: count, dtype: int64

MAD 20.000
SUBTYPE
BRCA_LumA      494
BRCA_LumB      193
BRCA_Basal     171
BRCA_Her2       77
BRCA_Normal     35
Name: count, dtype: int64


## Otimização por Grid Search

In [10]:
resultados_grid = []

for nome, df in datasets_pca.items():

    print("\n" + "=" * 60)
    print(f"EXECUTANDO GRID SEARCH: {nome}")
    print("=" * 60)

    X = df.to_numpy(dtype=float)

    y_true = labels_pam50[nome].to_numpy()

    resultados = run_grid_search(
        X=X,
        labels_true=y_true,
        reduction_grids=reduction_grids,
        clustering_grids=clustering_grids,
        random_state=2002,
        n_neighbors_metric=15,
        verbose = False
    )

    resultados["Cenario"] = nome

    resultados_grid.append(resultados)

# Concatenar resultados
df_grid_search = pd.concat(
    resultados_grid,
    ignore_index=True
)

print("\nDimensões dos resultados:")
print(df_grid_search.shape)

display(df_grid_search.head())


EXECUTANDO GRID SEARCH: MAD 3.000

EXECUTANDO GRID SEARCH: MAD 5.000

EXECUTANDO GRID SEARCH: MAD 10.000

EXECUTANDO GRID SEARCH: MAD 20.000

Dimensões dos resultados:
(6300, 17)


,Reduction,Clustering,Reduction_n_neighbors,Reduction_MN_ratio,Reduction_FP_ratio,Clustering_min_cluster_size,Clustering_min_samples,N_Clusters,N_Noise,Trustworthiness,Silhouette,ARI,NMI,Clustering_n_neighbors,Clustering_resolution,Reduction_min_dist,Cenario
0,PaCMAP,HDBSCAN,10,0.5,2.0,10.0,1.0,2,0,0.906481,0.796459,0.334395,0.48125,NaN,NaN,NaN,MAD 3.000
1,PaCMAP,HDBSCAN,10,0.5,2.0,10.0,5.0,2,0,0.906481,0.796459,0.334395,0.48125,NaN,NaN,NaN,MAD 3.000
2,PaCMAP,HDBSCAN,10,0.5,2.0,10.0,10.0,2,0,0.906481,0.796459,0.334395,0.48125,NaN,NaN,NaN,MAD 3.000
3,PaCMAP,HDBSCAN,10,0.5,2.0,10.0,15.0,2,0,0.906481,0.796459,0.334395,0.48125,NaN,NaN,NaN,MAD 3.000
4,PaCMAP,HDBSCAN,10,0.5,2.0,10.0,20.0,2,0,0.906481,0.796459,0.334395,0.48125,NaN,NaN,NaN,MAD 3.000


In [ ]:
resultados_grid = []

for nome, df in datasets_pca.items():

    print("\n" + "=" * 60)
    print(f"EXECUTANDO GRID SEARCH: {nome}")
    print("=" * 60)

    X = df.to_numpy(dtype=float)
    y_true = labels_pam50[nome].to_numpy()

    resultados = run_grid_search(
        X=X,
        labels_true=y_true,
        reduction_grids=reduction_grids,
        clustering_grids=clustering_grids,
        random_state=2002,
        n_neighbors_metric=15,
        verbose=False
    )

    resultados["Cenario"] = nome
    resultados_grid.append(resultados)

# Concatenar resultados
if resultados_grid:

    df_grid_search = pd.concat(
        resultados_grid,
        ignore_index=True
    )

    print("\nDimensões:", df_grid_search.shape)

    display(df_grid_search.head())

else:
    print("Nenhum resultado foi gerado.")

Dimensões: (6300, 17)


,Reduction,Clustering,Reduction_n_neighbors,Reduction_MN_ratio,Reduction_FP_ratio,Clustering_min_cluster_size,Clustering_min_samples,N_Clusters,N_Noise,Trustworthiness,Silhouette,ARI,NMI,Clustering_n_neighbors,Clustering_resolution,Reduction_min_dist,Cenario
0,PaCMAP,HDBSCAN,10,0.5,2.0,10.0,1.0,2,0,0.906707,0.792773,0.336059,0.480655,NaN,NaN,NaN,MAD 3.000
1,PaCMAP,HDBSCAN,10,0.5,2.0,10.0,5.0,2,0,0.906707,0.792773,0.336059,0.480655,NaN,NaN,NaN,MAD 3.000
2,PaCMAP,HDBSCAN,10,0.5,2.0,10.0,10.0,2,0,0.906707,0.792773,0.336059,0.480655,NaN,NaN,NaN,MAD 3.000
3,PaCMAP,HDBSCAN,10,0.5,2.0,10.0,15.0,2,0,0.906707,0.792773,0.336059,0.480655,NaN,NaN,NaN,MAD 3.000
4,PaCMAP,HDBSCAN,10,0.5,2.0,10.0,20.0,2,0,0.906707,0.792773,0.336059,0.480655,NaN,NaN,NaN,MAD 3.000


## Leiden no PCA

In [14]:
resultados_leiden_pca = []

for nome, df in datasets_pca.items():

    print(f"\nExecutando Leiden diretamente no PCA: {nome}")

    X = df.to_numpy(dtype=float)
    y_true = labels_pam50[nome].to_numpy()

    for n_neighbors in param_grid_leiden["n_neighbors"]:

        for resolution in param_grid_leiden["resolution"]:

            try:

                labels_pred, modelo = run_leiden(
                    X,
                    n_neighbors=n_neighbors,
                    resolution=resolution,
                    seed=2002,
                    weight_mode="none"
                )

                n_clusters = len(np.unique(labels_pred))

                # Silhouette
                if 2 <= n_clusters < len(labels_pred):

                    silhouette = silhouette_score(
                        X,
                        labels_pred
                    )

                else:
                    silhouette = np.nan

                # Métricas externas
                ari = adjusted_rand_score(
                    y_true,
                    labels_pred
                )

                nmi = normalized_mutual_info_score(
                    y_true,
                    labels_pred
                )

                resultados_leiden_pca.append({

                    "Cenario": nome,
                    "Metodo_Reducao": "PCA_50D",
                    "Metodo_Clustering": "Leiden",
                    "n_neighbors": n_neighbors,
                    "resolution": resolution,
                    "N_Clusters": n_clusters,
                    "N_Noise": 0,
                    "Trustworthiness": np.nan,
                    "Silhouette": silhouette,
                    "ARI": ari,
                    "NMI": nmi

                })

            except Exception as erro:

                print(
                    f"Erro em {nome}, "
                    f"k={n_neighbors}, "
                    f"resolution={resolution}: {erro}"
                )

df_leiden_pca = pd.DataFrame(
    resultados_leiden_pca
)

display(df_leiden_pca.head())



Executando Leiden diretamente no PCA: MAD 3.000

Executando Leiden diretamente no PCA: MAD 5.000

Executando Leiden diretamente no PCA: MAD 10.000

Executando Leiden diretamente no PCA: MAD 20.000


,Cenario,Metodo_Reducao,Metodo_Clustering,n_neighbors,resolution,N_Clusters,N_Noise,Trustworthiness,Silhouette,ARI,NMI
0,MAD 3.000,PCA_50D,Leiden,10,0.05,2,0,NaN,0.225308,0.328068,0.478108
1,MAD 3.000,PCA_50D,Leiden,10,0.10,2,0,NaN,0.225308,0.328068,0.478108
2,MAD 3.000,PCA_50D,Leiden,10,0.20,2,0,NaN,0.208227,0.429625,0.491591
3,MAD 3.000,PCA_50D,Leiden,10,0.30,3,0,NaN,0.110310,0.565931,0.557516
4,MAD 3.000,PCA_50D,Leiden,10,0.40,3,0,NaN,0.108705,0.575360,0.567003


## Salvando os resultados

In [15]:
# Resultados PaCMAP/UMAP + HDBSCAN/Leiden
PATH_GRID = RESULTS_DIR / "grid_search_resultados2.csv"

df_grid_search.to_csv(
    PATH_GRID,
    index=False
)

# Resultados Leiden diretamente no PCA
PATH_LEIDEN_PCA = RESULTS_DIR / "leiden_direto_pca2.csv"

df_leiden_pca.to_csv(
    PATH_LEIDEN_PCA,
    index=False
)

print("Resultados salvos!")
print(PATH_GRID)
print(PATH_LEIDEN_PCA)

Resultados salvos!
d:\Projetos Python\Mestrado\RBras - Copia\results\grid_search_resultados2.csv
d:\Projetos Python\Mestrado\RBras - Copia\results\leiden_direto_pca2.csv


## Seleção dos Melhores Modelos

In [16]:
from importlib import reload
import src.experiments as experiments

reload(experiments)

best_models = experiments.select_best_models(
    results=df_grid_search,
    metric="Silhouette",
    output_dir=RESULTS_DIR / "best_models2"
)

display(best_models)

Resultados salvos em: d:\Projetos Python\Mestrado\RBras - Copia\results\best_models2


,Reduction,Clustering,Reduction_n_neighbors,Reduction_MN_ratio,Reduction_FP_ratio,Clustering_min_cluster_size,Clustering_min_samples,N_Clusters,N_Noise,Trustworthiness,Silhouette,ARI,NMI,Clustering_n_neighbors,Clustering_resolution,Reduction_min_dist,Cenario
0,PaCMAP,HDBSCAN,50,0.5,2.0,15.0,15.0,2,0,0.904420,0.839749,0.335787,0.487499,NaN,NaN,NaN,MAD 3.000
1,PaCMAP,Leiden,50,0.5,2.0,NaN,NaN,2,0,0.904420,0.839749,0.335787,0.487499,30.0,0.05,NaN,MAD 3.000
2,UMAP,HDBSCAN,15,NaN,NaN,75.0,20.0,2,0,0.916796,0.832842,0.332742,0.482116,NaN,NaN,0.01,MAD 3.000
3,UMAP,Leiden,15,NaN,NaN,NaN,NaN,2,0,0.916796,0.832842,0.332742,0.482116,30.0,0.05,0.01,MAD 3.000
4,PaCMAP,HDBSCAN,50,0.5,2.0,15.0,10.0,2,0,0.900325,0.831373,0.335787,0.487499,NaN,NaN,NaN,MAD 5.000
5,PaCMAP,Leiden,50,0.5,2.0,NaN,NaN,2,0,0.900325,0.831373,0.335787,0.487499,50.0,0.10,NaN,MAD 5.000
6,UMAP,HDBSCAN,15,NaN,NaN,10.0,5.0,2,0,0.920296,0.823195,0.325049,0.473064,NaN,NaN,0.01,MAD 5.000
7,UMAP,Leiden,15,NaN,NaN,NaN,NaN,2,0,0.920296,0.823195,0.325049,0.473064,50.0,0.05,0.01,MAD 5.000
8,PaCMAP,HDBSCAN,50,0.5,3.0,30.0,5.0,2,0,0.912121,0.817311,0.345221,0.497479,NaN,NaN,NaN,MAD 10.000
9,PaCMAP,Leiden,50,0.5,3.0,NaN,NaN,2,0,0.912121,0.817311,0.345221,0.497479,50.0,0.05,NaN,MAD 10.000
